# Lezione 12. Casi d'uso reali: dai giochi classici ad AlphaStar

**DIH DATalife — Corso di Reinforcement Learning (RL) da zero**

Benvenuta o benvenuto all'ultima lezione del corso. Hai costruito un edificio in 11 lezioni: sei partita dal dilemma tra esplorare e sfruttare nei bandit, sei passata ai processi decisionali di Markov (PDM) e alla programmazione dinamica, hai imparato a predire e controllare senza modello con Monte Carlo, Differenza Temporale (TD), SARSA e Q-learning, hai fatto il salto alle reti neurali con le reti Q profonde (DQN, dall'inglese *Deep Q-Network*) e ai gradienti di politica, e hai chiuso con pianificazione, ricerca ad albero di Monte Carlo (MCTS, dall'inglese *Monte Carlo Tree Search*) e architetture di agenti distribuite. Tutto questo, finora, è stato in sostanza teoria da lavagna: formule, pseudocodice, ambienti giocattolo.

Questa lezione finale ti mostra che quella teoria non è rimasta sulla lavagna. Percorrerai una linea di sistemi reali, da Deep Blue ad AlphaStar, e in ciascuno riconoscerai pezzi che già conosci: una funzione di valore qui, una politica appresa con il gradiente là, una ricerca di tipo MCTS più avanti. Non memorizzare nomi né date: osserva come le stesse idee semplici, combinate e scalate, siano bastate a battere i migliori giocatori umani del pianeta in giochi che per decenni —o per secoli— erano stati considerati roccaforti dell'intelletto umano.

## Scaletta della lezione

1. Da Deep Blue alla Monte Carlo Tree Search (MCTS)
2. AlphaGo: la mossa che sorprese il mondo
3. AlphaZero: imparare da zero, senza dati umani
4. AlphaStar: la sfida di StarCraft II
5. Codice dal vivo: mini-AlphaZero per il Tris, cella per cella
6. Punti chiave e chiusura del corso

**Acronimi che incontrerai oggi** (nome per esteso la prima volta che compaiono, e qui in ordine alfabetico per consultazione rapida): APM (azioni al minuto), DQN (*Deep Q-Network*, reti Q profonde), GPI (Iterazione di Politica Generalizzata), IA (intelligenza artificiale), LSTM (*Long Short-Term Memory*, memoria a lungo e breve termine), MCTS (*Monte Carlo Tree Search*), PDM (processi decisionali di Markov), PPO (Ottimizzazione della Politica Prossimale), PUCT (*Predictor + UCT*), RL (*Reinforcement Learning*, apprendimento per rinforzo), TD (Differenza Temporale), TPU (unità di elaborazione tensoriale), UCT (*Upper Confidence bound for Trees*).


## 1. Da Deep Blue alla Monte Carlo Tree Search

*Traccia della lezione: inizia chiedendo chi ha sentito parlare di Deep Blue o di AlphaGo. Quasi sempre qualcuno conosce il nome ma non il meccanismo interno; oggi colleghi quel meccanismo a ciò che già sai.*

Prima che una qualsiasi rete neurale giocasse a Go, l'intelligenza artificiale (IA) batteva da 30 anni campioni umani nei giochi da tavolo, con tecniche molto più vicine alla ricerca che all'apprendimento. I giochi funzionano come "moscerini della frutta" per la ricerca in IA: regole chiare, obiettivi misurabili, un punteggio che non mente, così come la genetica usò la Drosophila per scoprire principi poi generalizzati a organismi molto più complessi.

**Chinook**, un programma per la dama, fu la prima tappa: raggiunse una partita praticamente perfetta combinando la potatura alfa-beta con l'analisi retrograda (risolvere i finali all'indietro, a partire da posizioni di vittoria già note). Il suo avversario fu Marion Tinsley, probabilmente il miglior giocatore di dama della storia: perse solo cinque partite in tutta la carriera, e due di queste furono contro Chinook. Successivamente, Schaeffer e il suo team risolsero il gioco per intero.

Nel 1997 arrivò **Deep Blue**, di IBM, che sconfisse il campione del mondo di scacchi Garri Kasparov in una sfida che divenne uno degli eventi più seguiti di internet dell'epoca. Deep Blue non imparava nulla: valutava circa 8.000 caratteristiche codificate a mano (sicurezza del re, struttura dei pedoni, materiale) ed esaminava fino a 200 milioni di posizioni al secondo, guardando tra 16 e 40 mosse in avanti. È, in sostanza, un minimax con potatura alfa-beta: forza bruta guidata da una funzione di valutazione progettata da esseri umani, senza traccia di apprendimento per rinforzo (RL, dall'inglese *Reinforcement Learning*).

Il contrappunto arrivò con **TD-Gammon**, un programma di backgammon che apprese una funzione di valore —la probabilità di vincere da una data posizione— addestrando una piccola rete neurale tramite TD, giocando solo contro se stesso, senza alcun dato umano. È la stessa idea che già conosci dalla predizione senza modello con TD e dall'approssimazione di funzioni di valore con una rete neurale, applicata quasi un decennio prima che esistesse il termine "deep learning". Curiosità: il backgammon si gioca con i dadi, e quella casualità forniva già un'esplorazione naturale, quindi TD-Gammon non ebbe bisogno di una strategia di esplorazione esplicita come l'ε-greedy dei bandit.

Poco dopo, **Maven** sconfisse il campione del mondo di Scrabble, Adam Logan, per nove partite a cinque, combinando una ricerca Monte Carlo troncata (simulare solo una decina di mosse in avanti, non partite complete) con una funzione di valutazione appresa. Quella ricerca Monte Carlo troncata è l'anello che collega tutto ciò che precede con la **MCTS**: invece di risolvere l'intero gioco, simuli molte continuazioni possibili dalla posizione attuale e apprendi una politica locale, specifica per quella situazione. Riconosci il ciclo di selezione, espansione, simulazione e retropropagazione della lezione sulla pianificazione: qui lo ritrovi come il ponte esatto verso AlphaGo.


## 2. AlphaGo: la mossa che sorprese il mondo

*Traccia della lezione: prima di spiegare l'architettura, chiedi perché la forza bruta di Deep Blue non funzionerebbe per il Go. Lascia che la classe arrivi da sola alla dimensione dell'albero di ricerca.*

Il Go si giocava da 3.000 anni e aveva circa 40 milioni di giocatori attivi quando DeepMind decise di affrontarlo. La difficoltà non stava solo nella sua scacchiera 19 per 19: ogni posizione ammette in media circa 200 mosse legali (contro le 20-35 degli scacchi), e il numero di configurazioni possibili supera quello degli atomi nell'universo osservabile. La forza bruta di Deep Blue, qui, semplicemente non era un'opzione; il Go si gioca più con l'intuizione che con il calcolo esaustivo, e AlphaGo doveva approssimare quell'intuizione.

L'architettura combinava due reti neurali convoluzionali: una **rete di politica**, che propone una distribuzione di probabilità sulle mosse legali, e una **rete di valore**, che stima un solo numero —la probabilità di vincere— da una data posizione. Riconosci entrambi i pezzi: la rete di valore è, in sostanza, la stessa funzione di valore che già conosci, appresa con una rete neurale; la rete di politica è un gradiente di politica, addestrato prima per imitazione e poi tramite autogioco. Le due reti si combinavano con la MCTS attraverso una variante chiamata PUCT (*Predictor + UCT*, dove UCT è *Upper Confidence bound for Trees*): la politica riduce l'ampiezza della ricerca (quali mosse vale la pena esplorare) e la rete di valore ne riduce la profondità (evita di giocare ogni ramo fino alla fine). Con questo, AlphaGo arrivava a guardare tra 50 e 60 mosse in avanti.

L'addestramento ebbe due fasi: prima, apprendimento supervisionato a partire da partite di giocatrici e giocatori umani esperti, perché la rete di politica imitasse mosse forti; poi, raffinamento tramite autogioco, giocando milioni di partite contro versioni precedenti di se stessa, mentre la rete di valore imparava a prevedere il risultato finale di ogni posizione raggiunta. In circa 72 ore di addestramento, AlphaGo passò dal giocare come un principiante a un livello sovrumano.

I risultati parlano da soli: AlphaGo sconfisse la campionessa europea Fan Hui per cinque partite a zero, e nel marzo 2016, a Seul, batté il leggendario Lee Sedol —9 dan, 18 titoli mondiali— per quattro partite a una. La mossa più ricordata fu la **mossa 37** della seconda partita: un colpo di spalla sulla quinta linea così insolito che la stessa AlphaGo calcolò una probabilità di appena una su 10.000 che un umano la giocasse, e che alla fine si rivelò una genialità strategica. Ma il sistema ebbe anche un limite: nella partita quattro, Lee Sedol trovò la **mossa 78**, a cui AlphaGo aveva assegnato appena lo 0,007% di probabilità, e vinse quella partita. Fu un promemoria utile e onesto: "sovrumano" non significa infallibile. Mesi dopo, AlphaGo Zero —addestrata da zero, senza alcun dato umano— batté il 90% delle partite contro AlphaGo Master, la versione che a sua volta aveva accumulato 60 vittorie e nessuna sconfitta contro i migliori professionisti del mondo.


## 3. AlphaZero: imparare da zero, senza dati umani

*Traccia della lezione: scrivi alla lavagna la formula di miglioramento della politica e chiedi quale ruolo svolge ciascun simbolo prima di proseguire nella lettura. È lo stesso GPI della lezione sul controllo senza modello, sotto mentite spoglie.*

AlphaZero semplifica e generalizza tutto ciò che precede in un colpo solo: una sola rete neurale, con output congiunto di politica e valore, e un unico algoritmo applicato senza modifiche a scacchi, shōgi (gli scacchi giapponesi) e Go, cambiando solo la codifica della scacchiera. Nessun dato umano, nessuna apertura o finale precaricati: parte da pesi casuali e dalle regole del gioco, e migliora esclusivamente giocando contro se stessa.

Il meccanismo che chiude il ciclo è puro GPI —l'Iterazione di Politica Generalizzata, la stessa alternanza di valutare e migliorare che hai visto con SARSA e Q-learning—, solo che qui la MCTS fa il ruolo di "operatore di miglioramento della politica": la ricerca produce una distribuzione di mosse migliore di quella che la rete suggerirebbe da sola, e la rete viene poi addestrata a imitare quella distribuzione migliorata. Formalmente:

π′(a|s) ∝ Q(s,a)^(1/τ)

dove π′ è la politica migliorata, Q(s,a) è il valore stimato dalla MCTS per ciascuna mossa e τ (tau) è un parametro di temperatura: valori alti esplorano di più, valori bassi concentrano la scelta sulla mossa più promettente. La rete si addestra con una perdita di entropia incrociata per la parte di politica e con errore quadratico medio per la parte di valore, tutto in un'unica rete.

I risultati, con hardware specializzato (unità di elaborazione tensoriale, TPU), arrivarono con una velocità che sorprese perfino le sue creatrici e i suoi creatori: dopo appena circa quattro ore di autogioco, AlphaZero superava Stockfish, il motore di scacchi più forte del momento; dopo circa due ore, superava Elmo, il miglior programma di shōgi; e dopo circa otto ore, superava la stessa versione di AlphaGo che aveva battuto Lee Sedol. Contro Stockfish —che valuta milioni di posizioni al secondo con regole di materiale fissate a mano (pedone = 1, cavallo = alfiere = 3, torre = 5, donna = 9)— AlphaZero sviluppò uno stile proprio: sacrificare materiale in cambio di attività dei pezzi e controllo a lungo termine. In una partita documentata con apertura Ruy López, AlphaZero sacrificò la donna alla mossa 25 per smantellare la struttura di Stockfish e finì per vincere con scacco matto; in un'altra, con apertura siciliana, sacrificò una torre per un pedone alla mossa 24 per assicurarsi una struttura di pedoni superiore. Nessun manuale di scacchi le aveva insegnato a farlo: lo scoprì giocando contro se stessa, nello stesso modo in cui il Q-learning apprende una politica ottimale senza che qualcuno le dica in anticipo quale sia.


## 4. AlphaStar: la sfida di StarCraft II

*Traccia della lezione: chiedi alla classe di elencare cosa rende StarCraft II diverso dagli scacchi o dal Go prima di leggere la risposta. Di solito emerge "non si vede tutta la mappa" e "bisogna decidere in tempo reale", che è proprio il punto.*

Se il Go e gli scacchi sono giochi a informazione perfetta e a turni discreti, StarCraft II è esattamente l'opposto: informazione parziale (la famosa "nebbia di guerra" nasconde gran parte della mappa), milioni di azioni possibili in ogni istante e decisioni da prendere in tempo reale, non a turni, combinando pianificazione strategica a lungo termine con la micro-gestione di decine di unità contemporaneamente. DeepMind costruì **AlphaStar** per giocare con la razza Protoss in questo ambiente, in collaborazione con Blizzard Entertainment.

L'addestramento combinò due fasi che già riconosci: prima, circa tre giorni di apprendimento per imitazione su partite umane reali, che portarono l'agente a un livello approssimativo di giocatore "platino"; poi, circa una settimana di autogioco organizzato nella cosiddetta **Lega AlphaStar** —agenti principali, "sfruttatori" specializzati nel trovare le loro debolezze e agenti di prova, che competono tra loro a un ritmo equivalente a centinaia di anni di partite umane—. Questo schema a lega con molti agenti che giocano in parallelo è la stessa filosofia distribuita dei metodi actor-critic distribuiti che hai visto, portata a una scala molto più grande; e l'algoritmo di RL che regolava la politica a ogni partita fu l'Ottimizzazione della Politica Prossimale (PPO, dall'inglese *Proximal Policy Optimization*), un metodo di gradiente di politica che limita quanto la politica può cambiare a ogni aggiornamento, per non destabilizzare l'apprendimento.

Architetturalmente, AlphaStar combinò reti convoluzionali per elaborare la mappa, memoria a lungo e breve termine (LSTM, dall'inglese *Long Short-Term Memory*) per ricordare informazioni passate sotto osservazione parziale, e meccanismi di attenzione per dare priorità alle unità rilevanti in ogni momento. Per rendere il confronto con gli umani equo, il team limitò il suo tempo di reazione a circa 350 millisecondi e le sue azioni al minuto (APM) a livelli comparabili a quelli di una giocatrice o un giocatore professionista, in modo che una vittoria riflettesse decisioni migliori e non una velocità di clic sovrumana. Il risultato: una vittoria per dieci a zero contro il professionista Grzegorz "MaNa" Komincz, dopo essersi misurato anche contro Liquid TLO.

**Osserva lo schema che si ripete nei quattro sistemi**: esplorazione/sfruttamento su scala di lega, autogioco che genera i propri dati e gradienti di politica addestrati con PPO per digerire tutto questo, stavolta sotto informazione incompleta. Nulla di tutto questo è magia nuova: sono gli stessi ingredienti di sempre, combinati con più calcolo e più pazienza.


## 5. Codice dal vivo: mini-AlphaZero per il Tris

*Traccia della lezione: è il blocco centrale della lezione. Costruisci il codice cella per cella, senza anticipare la cella del ciclo principale. Prima di eseguire ogni dimostrazione di autogioco, proietta lo schermo e chiedi alla classe di prevedere la prossima mossa.*

Implementerai, in scala giocattolo, lo stesso schema che hai appena visto in AlphaGo, AlphaZero e AlphaStar: autogioco, ricerca di tipo MCTS e miglioramento iterativo della politica. Giocherai a Tris invece che al Go, e userai una semplice funzione euristica più una tabella di valori al posto di una rete neurale addestrata davvero, così da poterlo eseguire in pochi secondi su qualsiasi Google Colab, senza altra dipendenza che `numpy`.

**Avvertenza didattica**: questa è una versione giocattolo dello schema di AlphaZero. Non è AlphaZero vero: non c'è una rete neurale convoluzionale addestrata con discesa del gradiente, il "valore" è memorizzato in un dizionario invece che in una rete, e il gioco è il Tris, non gli scacchi né il Go. Lo scopo è farti vedere lo stesso ingranaggio —autogioco, ricerca, miglioramento della politica— funzionare dall'inizio alla fine su un gioco banale, non riprodurre il sistema reale.

**Esegui ogni cella in ordine.** Troverai, ogni poche celle, una partita di autogioco stampata mossa per mossa: proiettala e commentala con la classe prima di proseguire.


In [ ]:
import math
import random

import numpy as np

np.random.seed(0)
random.seed(0)


### La scacchiera

La scacchiera è una tupla di nove caselle: `0` vuota, `1` una pedina X, `-1` una pedina O. `LINEAS_GANADORAS` elenca le otto combinazioni di tre caselle che fanno vincere la partita.


In [ ]:
LINEAS_GANADORAS = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6),
]


def tablero_nuevo():
    return tuple([0] * 9)  # 0 = vacio, 1 = X, -1 = O


def movimientos_validos(tablero):
    return [i for i, v in enumerate(tablero) if v == 0]


def aplicar_movimiento(tablero, pos, jugador):
    nuevo = list(tablero)
    nuevo[pos] = jugador
    return tuple(nuevo)


### Il risultato di una partita

`resultado` scorre le otto linee vincenti e controlla se una qualsiasi somma +3 (vince X) o -3 (vince O). Se la scacchiera è piena e nessuno ha vinto, è pareggio. Se nulla di tutto questo accade, la partita prosegue.


In [ ]:
def resultado(tablero):
    """Devuelve +1 si gana X, -1 si gana O, 0 si hay empate, None si sigue."""
    for a, b, c in LINEAS_GANADORAS:
        suma = tablero[a] + tablero[b] + tablero[c]
        if suma == 3:
            return 1
        if suma == -3:
            return -1
    if 0 not in tablero:
        return 0
    return None


### Un aiuto solo per la lezione: stampare la scacchiera

Questa funzione non fa parte dell'algoritmo: la aggiungiamo solo per poter proiettare le partite di autogioco durante la lezione, casella per casella.


In [ ]:
def imprimir_tablero(tablero):
    """Imprime el tablero en tres filas, con X, O y punto para las casillas vacias.
    Solo para visualizar en clase: el algoritmo nunca llama a esta funcion."""
    simbolos = {1: "X", -1: "O", 0: "."}
    for f in range(3):
        fila = tablero[f * 3:(f + 1) * 3]
        print(" ".join(simbolos[v] for v in fila))
    print()


imprimir_tablero(tablero_nuevo())


### L'euristica e la tabella dei valori

`heuristica` è il sostituto minimo della rete di valore di AlphaZero: per ogni linea di tre caselle, se contiene solo pedine di X (o nessuna) somma le proprie pedine al quadrato; se contiene solo pedine di O, sottrae lo stesso valore. È una funzione fissa, non appresa: serve solo come stima di partenza quando la tabella dei valori non ha ancora visto una posizione. Restituisce un valore approssimativo in [-1, 1] dal punto di vista di X.

`tabla_valores` è un dizionario, `stato -> valore stimato`, che svolge, in scala minima, il ruolo che in AlphaZero svolge la rete di valore addestrata: la riempirai con l'autogioco tra poco.


In [ ]:
def heuristica(tablero):
    """Sustituta minima de la red de valor de AlphaZero. Devuelve un valor
    aproximado en [-1, 1] desde la perspectiva de X."""
    puntuacion = 0
    for linea in LINEAS_GANADORAS:
        valores = [tablero[i] for i in linea]
        piezas_x = valores.count(1)
        piezas_o = valores.count(-1)
        if piezas_o == 0:
            puntuacion += piezas_x ** 2
        if piezas_x == 0:
            puntuacion -= piezas_o ** 2
    return math.tanh(puntuacion / 6.0)


# Tabla de valores aprendida por autojuego: estado -> valor estimado
# desde la perspectiva de X (+1 = X gana, -1 = O gana, 0 = tablas).
tabla_valores = {}


def valor_estimado(tablero):
    if tablero in tabla_valores:
        return tabla_valores[tablero]
    return heuristica(tablero)


### Il nodo dell'albero di ricerca

Ogni `Nodo` memorizza la propria scacchiera, di chi è il turno, i figli già esplorati, quante volte è stato visitato (`N`) e la somma cumulativa dei valori che ha ricevuto (`W`). `valor_medio` è semplicemente `W / N`: la media di tutte le simulazioni passate da questo nodo.


In [ ]:
class Nodo:
    __slots__ = ("tablero", "jugador", "hijos", "N", "W", "expandido")

    def __init__(self, tablero, jugador):
        self.tablero = tablero
        self.jugador = jugador  # jugador (+1 o -1) que debe mover aqui
        self.hijos = {}
        self.N = 0
        self.W = 0.0
        self.expandido = False

    def valor_medio(self):
        return self.W / self.N if self.N > 0 else 0.0


### UCT: come decidere quale ramo esplorare

`ucb_score` combina "quanto sembra buono" un figlio (`q`, convertito dal punto di vista del giocatore che muove nel padre) con "quanto è stato esplorato" (`u`, che cresce se il figlio è stato visitato poco). È la stessa logica di Upper Confidence Bound (UCB) che hai già usato con i bandit, ora applicata a un albero: per questo si chiama UCT (*Upper Confidence bound for Trees*). In AlphaZero, `u` incorpora anche la probabilità a priori della rete di politica (è esattamente ciò che aggiunge PUCT); qui usiamo un'esplorazione uniforme per mantenere semplice l'esempio.


In [ ]:
def ucb_score(padre, hijo, c=1.4):
    q = hijo.valor_medio() * padre.jugador
    u = c * math.sqrt(math.log(padre.N + 1) / (hijo.N + 1))
    return q + u


### Il ciclo della MCTS: selezione, espansione, valutazione e retropropagazione

`simular` è il cuore dell'esempio. Ogni chiamata scende nell'albero fino a trovare un nodo non espanso, lo espande creando un figlio per ogni mossa valida, valuta quella nuova posizione e propaga il valore verso l'alto aggiornando `N` e `W` di ogni nodo del percorso.

Fai attenzione a un dettaglio importante: invece di un *rollout* casuale fino alla fine della partita (la MCTS "classica" che hai visto nella lezione sulla pianificazione), valutiamo la posizione direttamente con `valor_estimado` —la tabella dei valori o, se non la conosce ancora, l'euristica—. È esattamente ciò che fa AlphaZero: sostituisce il *rollout* con la sua rete di valore.


In [ ]:
def simular(nodo):
    """Un ciclo de seleccion, expansion, evaluacion y retropropagacion."""
    r = resultado(nodo.tablero)
    if r is not None:
        nodo.N += 1
        nodo.W += r
        return r

    if not nodo.expandido:
        # --- Expansion: un hijo por cada movimiento valido ---
        for m in movimientos_validos(nodo.tablero):
            tablero_hijo = aplicar_movimiento(nodo.tablero, m, nodo.jugador)
            nodo.hijos[m] = Nodo(tablero_hijo, -nodo.jugador)
        nodo.expandido = True
        # --- Evaluacion con la tabla de valores/heuristica, en vez de rollout ---
        v = valor_estimado(nodo.tablero)
        nodo.N += 1
        nodo.W += v
        return v

    # --- Seleccion: bajamos por el hijo con mejor puntuacion UCT ---
    mejor_m = max(nodo.hijos, key=lambda m: ucb_score(nodo, nodo.hijos[m]))
    v = simular(nodo.hijos[mejor_m])
    # --- Retropropagacion: el resultado sube y actualiza este nodo tambien ---
    nodo.N += 1
    nodo.W += v
    return v


### Lanciare molte simulazioni da una posizione

`busqueda_mcts` crea la radice per la scacchiera e il giocatore dati, chiama `simular` `num_simulaciones` volte, e restituisce quante visite ha ricevuto ciascuna mossa possibile da lì. Più visite significa che l'albero di ricerca la considera più promettente.


In [ ]:
def busqueda_mcts(tablero, jugador, num_simulaciones=60):
    """Devuelve {movimiento: visitas} tras num_simulaciones simulaciones."""
    raiz = Nodo(tablero, jugador)
    for _ in range(num_simulaciones):
        simular(raiz)
    return {m: hijo.N for m, hijo in raiz.hijos.items()}


**Prima dimostrazione: esegui la ricerca da una scacchiera vuota.** Con 60 simulazioni e X che muove per primo, osserva quale casella accumula più visite: se la ricerca funziona bene, dovrebbe essere la casella 4, il centro, perché partecipa a quattro linee vincenti diverse.


In [ ]:
visitas_demo = busqueda_mcts(tablero_nuevo(), 1, num_simulaciones=60)
print("Visitas por movimiento desde el tablero vacio:", visitas_demo)


### Dalle visite a una mossa: il miglioramento della politica

`elegir_movimiento` converte le visite della ricerca in una politica e sceglie una mossa. Con temperatura > 0 c'è esplorazione (utile nell'autogioco, per generare partite variate); con temperatura = 0 si gioca sempre la mossa più visitata (la modalità usata in valutazione). È la versione pratica di π′(a|s) ∝ Q(s,a)^(1/τ), la formula di miglioramento della politica che hai visto poco fa.


In [ ]:
def elegir_movimiento(visitas, temperatura):
    movimientos = list(visitas.keys())
    conteos = np.array([visitas[m] for m in movimientos], dtype=float)

    if temperatura == 0:
        return movimientos[int(np.argmax(conteos))]

    probabilidades = conteos ** (1.0 / temperatura)
    probabilidades /= probabilidades.sum()
    return np.random.choice(movimientos, p=probabilidades)


### Una partita completa di autogioco

`jugar_partida_autojuego` gioca X contro O, entrambi controllati dalla stessa ricerca MCTS: è l'autogioco vero e proprio. Memorizza ogni scacchiera visitata, pronta ad aggiornare la tabella dei valori non appena si conosce il risultato finale. Il parametro `mostrar`, che non esiste nel mini-AlphaZero "di produzione", serve solo a proiettare la partita in classe.


In [ ]:
def jugar_partida_autojuego(num_simulaciones=60, temperatura=0.8, mostrar=False):
    """Juega una partida completa de X contra O, ambos controlados por la
    misma busqueda MCTS (autojuego). Devuelve los tableros visitados y el
    resultado final, listos para actualizar la tabla de valores."""
    tablero = tablero_nuevo()
    jugador = 1
    historial = []
    turno = 1

    while resultado(tablero) is None:
        visitas = busqueda_mcts(tablero, jugador, num_simulaciones)
        movimiento = elegir_movimiento(visitas, temperatura)
        historial.append(tablero)
        tablero = aplicar_movimiento(tablero, movimiento, jugador)
        if mostrar:
            print(f"Turno {turno}: juega {'X' if jugador == 1 else 'O'} en la casilla {movimiento}")
            imprimir_tablero(tablero)
        jugador = -jugador
        turno += 1

    return historial, resultado(tablero)


**Seconda dimostrazione — proietta questa partita in classe.** La tabella dei valori è ancora vuota, quindi l'agente gioca solo con l'euristica fissa: è più o meno il livello di un AlphaZero appena inizializzato, prima della prima generazione di autogioco. *Traccia della lezione: fermati dopo ogni turno e chiedi se la mossa ti sembra ragionevole.*


In [ ]:
_, resultado_demo = jugar_partida_autojuego(mostrar=True)
print("Resultado:", {1: "Gana X", -1: "Gana O", 0: "Empate"}[resultado_demo])


### Il passo di miglioramento: aggiornare la tabella dei valori

`actualizar_tabla_valores` avvicina il valore stimato di ogni posizione visitata durante la partita al risultato reale, con un aggiornamento di tipo Monte Carlo / TD. Col tempo, la tabella smette di dipendere dall'euristica iniziale e riflette ciò che l'autogioco stesso ha appreso. Questo è, su scala minuscola, l'addestramento della rete di valore di AlphaZero.


In [ ]:
def actualizar_tabla_valores(historial, resultado_final, alpha=0.5):
    for tablero in historial:
        actual = valor_estimado(tablero)
        tabla_valores[tablero] = actual + alpha * (resultado_final - actual)


### Come misuriamo il progresso: partite contro un giocatore casuale

Per sapere se l'agente migliora davvero, lo facciamo giocare contro un avversario che muove a caso. `movimiento_aleatorio` è quell'avversario; `jugar_vs_aleatorio` gioca una partita di valutazione con l'agente MCTS (temperatura 0, cioè gioca sempre la sua mossa migliore secondo la tabella dei valori attuale), sempre come X.


In [ ]:
def movimiento_aleatorio(tablero):
    return random.choice(movimientos_validos(tablero))


def jugar_vs_aleatorio(num_simulaciones=60):
    tablero = tablero_nuevo()
    jugador = 1
    while resultado(tablero) is None:
        if jugador == 1:
            visitas = busqueda_mcts(tablero, jugador, num_simulaciones)
            movimiento = elegir_movimiento(visitas, temperatura=0)
        else:
            movimiento = movimiento_aleatorio(tablero)
        tablero = aplicar_movimiento(tablero, movimiento, jugador)
        jugador = -jugador
    return resultado(tablero)


### Il ciclo completo: autogioco + miglioramento, generazione dopo generazione

Qui il cerchio si chiude. In ogni generazione: prima, 40 partite di autogioco che alimentano la tabella dei valori; poi, 50 partite di valutazione contro il giocatore casuale, per misurare quanto è migliorato l'agente. Fissiamo di nuovo il seme appena prima di iniziare, così l'esperimento è riproducibile indipendentemente da quali celle dimostrative hai eseguito prima.

**Esegui la cella e osserva il punteggio di ogni generazione: è lo stesso ciclo di autogioco che ha addestrato AlphaGo, AlphaZero e AlphaStar, solo che qui richiede secondi invece di ore.**


In [ ]:
np.random.seed(0)
random.seed(0)

GENERACIONES = 6
PARTIDAS_AUTOJUEGO_POR_GENERACION = 40
PARTIDAS_EVALUACION = 50

for generacion in range(1, GENERACIONES + 1):
    # 1) Autojuego: generar partidas y usarlas para mejorar la tabla de valores
    for _ in range(PARTIDAS_AUTOJUEGO_POR_GENERACION):
        historial, resultado_final = jugar_partida_autojuego()
        actualizar_tabla_valores(historial, resultado_final)

    # 2) Evaluacion: medir el nivel actual contra un jugador aleatorio
    victorias, empates, derrotas = 0, 0, 0
    for _ in range(PARTIDAS_EVALUACION):
        r = jugar_vs_aleatorio()
        if r == 1:
            victorias += 1
        elif r == 0:
            empates += 1
        else:
            derrotas += 1

    print(
        f"Generacion {generacion}: "
        f"{victorias} victorias, {empates} empates, {derrotas} derrotas "
        f"(sobre {PARTIDAS_EVALUACION} partidas contra un jugador aleatorio) "
        f"| tabla de valores con {len(tabla_valores)} posiciones aprendidas"
    )


### Lettura del risultato

*Traccia della lezione: fermati qui e chiedi alla classe cosa si aspettava di vedere nella prima generazione rispetto all'ultima.*

In questa esecuzione di riferimento (con il seme fissato a 0), il punteggio è: generazione 1, 50 vittorie e 0 sconfitte; generazione 2, 49 vittorie, 1 pareggio e 0 sconfitte; generazione 3, 48 vittorie, 2 pareggi e 0 sconfitte; generazione 4, 50 vittorie e 0 sconfitte; generazione 5, 48 vittorie, 2 pareggi e 0 sconfitte; generazione 6, 47 vittorie, 3 pareggi e 0 sconfitte. L'agente non perde più nemmeno una partita fin dalla prima generazione, e il numero di pareggi e vittorie varia leggermente a seconda di quanto ha esplorato ogni tornata di autogioco. La tabella dei valori, nel frattempo, cresce in modo costante: passa da 196 posizioni apprese dopo la generazione 1 a 748 dopo la generazione 6.

Il Tris è così piccolo che lo schema converge quasi subito, cosa che non accade affatto con gli scacchi o il Go, dove AlphaZero ha avuto bisogno di ore di calcolo massiccio —con hardware specializzato— invece di secondi su un portatile. L'idea di fondo, però, è identica: autogioco che genera dati, una ricerca che migliora la politica locale, e un aggiornamento che riporta quel miglioramento nella conoscenza accumulata.


**Terza dimostrazione — proietta quest'ultima partita in classe.** L'agente ha ormai addestrato sei generazioni: confrontala con la prima dimostrazione, prima dell'addestramento, e chiedi alla classe di indicare la differenza.


In [ ]:
_, resultado_final_demo = jugar_partida_autojuego(temperatura=0.3, mostrar=True)
print("Resultado:", {1: "Gana X", -1: "Gana O", 0: "Empate"}[resultado_final_demo])


## Punti chiave

- Prima di AlphaGo, l'IA giocava già a livello sovrumano a dama, scacchi, backgammon e Scrabble, combinando ricerca (minimax, potatura alfa-beta) con funzioni di valore, a volte apprese e a volte progettate a mano
- La MCTS è il ponte tra quella storia e AlphaGo: cerca tramite simulazioni invece di risolvere l'intero gioco, e UCT decide quale ramo esplorare con la stessa logica di UCB nei bandit
- AlphaGo combinò una rete di politica, una rete di valore e la MCTS tramite PUCT, addestrate prima per imitazione di partite umane e poi tramite autogioco, e sconfisse Lee Sedol per quattro partite a una
- AlphaZero eliminò completamente i dati umani: una sola rete, un solo algoritmo, applicato senza modifiche a scacchi, shōgi e Go, superando Stockfish, Elmo e la stessa AlphaGo in poche ore di autogioco, guidato dal GPI e dalla formula π′(a|s) ∝ Q(s,a)^(1/τ)
- AlphaStar ha dimostrato che lo stesso schema —imitazione più autogioco, stavolta organizzato in una lega di agenti e regolato con PPO— funziona anche con informazione parziale e spazi di azione enormi, vincendo dieci a zero contro una professionista di StarCraft II
- Il mini-AlphaZero di questa lezione riproduce lo stesso ingranaggio —autogioco, ricerca di tipo MCTS, miglioramento della politica— su scala Tris: stesso schema, stesse componenti, senza bisogno di altro che `numpy`
- Ogni sistema che hai visto oggi è una ricombinazione di idee che già conoscevi: funzioni di valore, politiche apprese con il gradiente, controllo fuori politica, approssimazione con reti neurali, e ricerca di tipo MCTS guidata da quelle reti


## Prima di chiudere

**Apri il notebook di pratica di questa lezione e completa i due esercizi.** Programmerai tu stessa o tu stesso, da zero, il passo di selezione ed espansione della MCTS, e adatterai questo stesso scheletro —autogioco, ricerca, tabella dei valori— a una versione ridotta di Forza 4.

## E con questo, chiudiamo il corso

Hai iniziato questo corso con il dilemma più semplice possibile di un singolo bandit a due bracci, decidendo se esplorare o sfruttare. Termini vedendo come quella stessa tensione —esplorare contro sfruttare, valutare contro migliorare, agire contro pianificare— sia cresciuta fino a battere il miglior giocatore di Go del mondo, reinventare gli scacchi senza aiuto umano e vincere dieci a zero contro professionisti di un videogioco in tempo reale. L'idea di fondo non è cambiata in nessun momento del percorso: è cambiata la scala, il calcolo e la pazienza nel lasciare che l'autogioco facesse il suo lavoro.

Quello che hai percorso in queste 12 lezioni —bandit, PDM, programmazione dinamica, Monte Carlo e TD, SARSA e Q-learning, DQN, gradienti di politica, pianificazione, architetture di agenti distribuiti, e ora questi sistemi che hanno fatto storia— è una base solida per leggere gli articoli originali di DeepMind, sperimentare con librerie come Gymnasium o PettingZoo, o semplicemente continuare a giocare con le tue varianti del codice di questa lezione. Il campo continua a muoversi rapidamente, e il modo migliore per tenere il passo è quello che hai imparato qui: prova, osserva il risultato, correggi la politica e riprova.

**Complimenti per essere arrivata o arrivato fino alla fine del corso.** Ora tocca a te continuare a esplorare.
